In [1]:
import pandas as pd
import re

csv_df = pd.read_csv("GIS Dashboard.csv")

csv_df.columns = [
    'constituency', 'project_id', 'project_name', 'state',
    'districts', 'nh_number', 'length_km', 'lane_config',
    'work_type', 'contract_type', 'sanctioned_amt_cr',
    'physical_progress_pct', 'start_date', 'end_date',
    'piu_city', 'piu_contact', 'ro_name', 'ro_contact'
]

null_values = [
    'TBD', 'PC', '146346', 'nan', 'Greenfield',
    'NE 4', '65(New NH-52', 'Yet to be decided',
    'Old NH-24(New NH-30)', '30 (Old NH-7)',
    '27E.W.', '8A (Extn.)', 'RING ROAD', 'Outer Ring Road'
]

def clean_nh(val):
    val = str(val).strip()
    if val in null_values:
        return None
    val = re.sub(r'^N\s*H[-\s]*', '', val, flags=re.IGNORECASE).strip()
    val = re.split(r'\s*[,&/]\s*|\s+and\s+', val, flags=re.IGNORECASE)[0].strip()
    val = re.sub(r'(\d+)\s+([A-Za-z])', r'\1\2', val).strip()
    if val in null_values:
        return None
    return val if val else None

csv_df['nh_number_clean'] = csv_df['nh_number'].apply(clean_nh)

# null anything with brackets or dots remaining
csv_df.loc[csv_df['nh_number_clean'].str.contains(r'[\(\)\.]', na=False), 'nh_number_clean'] = None

# drop nulls
csv_df = csv_df[csv_df['nh_number_clean'].notna()].reset_index(drop=True)

print(f"{csv_df.shape[0]} clean rows remaining")

csv_df.to_csv("GIS_cleaned.csv", index=False)
print("Saved to GIS_cleaned.csv")

1301 clean rows remaining
Saved to GIS_cleaned.csv


In [1]:
import pandas as pd
piu_df = pd.read_csv('piu_geocoded.csv')
print(piu_df.shape)
print(piu_df[piu_df['lat'].isna()])

(199, 6)
                  piu_city   piu_contact             ro_name  ro_contact  lat  \
16          Prayagraj (WB)  8.130006e+09          RO-UP East  8130006065  NaN   
30             Vizianagram  8.130006e+09       RO-Vijayawada  8130006067  NaN   
84          Bhiwani-CHD-HR  8.130006e+09  RO-Chandigarh (HR)  8130006043  NaN   
118            CMU Mathura  8.130006e+09            RO-Delhi  8130006046  NaN   
124  Ludhiana (Expressway)  8.130006e+09  RO-Chandigarh (PB)  8130006043  NaN   
139    VasantViharDehradun  8.130006e+09         RO-Dehradun  8130006045  NaN   
157              Nagarcoil  8.130006e+09          RO-Madurai  8130006055  NaN   
160         Sonipat-CHD-HR  8.130006e+09  RO-Chandigarh (HR)  8130006043  NaN   

     lng  
16   NaN  
30   NaN  
84   NaN  
118  NaN  
124  NaN  
139  NaN  
157  NaN  
160  NaN  


In [2]:
import pandas as pd
csv_df = pd.read_csv("GIS_cleaned.csv")
print(csv_df.shape)
print(csv_df.columns.tolist())

(1301, 19)
['constituency', 'project_id', 'project_name', 'state', 'districts', 'nh_number', 'length_km', 'lane_config', 'work_type', 'contract_type', 'sanctioned_amt_cr', 'physical_progress_pct', 'start_date', 'end_date', 'piu_city', 'piu_contact', 'ro_name', 'ro_contact', 'nh_number_clean']


In [3]:
print(csv_df['nh_number_clean'].nunique())

299


In [5]:
from collections import Counter
import json
import re

with open("nh_enriched.geojson") as f:
    data = json.load(f)

def extract_nh_numbers(name):
    matches = re.findall(r'NH\s*([A-Za-z0-9]+)', name, re.IGNORECASE)
    return [m.strip().upper() for m in matches]

nh_segment_count = Counter()
for feature in data['features']:
    name = feature['properties'].get('Name', '')
    for nh in extract_nh_numbers(name):
        nh_segment_count[nh] += 1

print("top 10 NHs by segment count:")
for nh, count in nh_segment_count.most_common(10):
    print(f"  NH {nh}: {count} segments")

top 10 NHs by segment count:
  NH 48: 3209 segments
  NH 44: 3172 segments
  NH 27: 2486 segments
  NH 16: 2199 segments
  NH 52: 1501 segments
  NH 53: 1232 segments
  NH 66: 1020 segments
  NH 19: 911 segments
  NH 30: 815 segments
  NH 65: 786 segments


In [6]:
import json

with open("nh_enriched.geojson") as f:
    data = json.load(f)

for feature in data['features'][:3]:
    print(f"NH: {feature['properties']['Name']}")
    print(f"projects: {len(feature['properties']['projects'])}")
    print(f"geometry type: {feature['geometry']['type']}")
    print()

NH: NH 44
projects: 75
geometry type: MultiLineString

NH: NH 66
projects: 34
geometry type: MultiLineString

NH: NH 944
projects: 1
geometry type: MultiLineString



In [7]:
import pandas as pd

officers_df = pd.read_csv('nhai_officers.csv')
piu_df = pd.read_csv('piu_geocoded.csv')

# check what designations exist
print(officers_df['designation'].value_counts().head(20))

# check RO names in piu
print(piu_df['ro_name'].unique())

designation
Manager                                       193
Deputy Manager                                150
Deputy General Manager & Project Director      90
Deputy General Manager                         49
General Manager & Project Director             45
Project Director                               23
General Manager                                16
Chief General Manager/Regional Officer         12
General Manager & Regional Officer             10
Deputy General Manager /PD                      5
Deputy General Manager &  Project Director      3
General Manager/PD                              2
Chief General Manager                           2
Name: count, dtype: int64
<StringArray>
[         'RO-Hyderabad',        'RO-Gandhinagar',             'RO-Mumbai',
             'RO-Jaipur',            'RO-UP West',             'RO-Nagpur',
 'RO-Thiruvananthapuram',            'RO-Kolkata',            'RO-UP East',
           'RO-Dehradun',         'RO-Vijayawada',    'RO-Chandigarh (

In [8]:
import pandas as pd
import re

officers_df = pd.read_csv('nhai_officers.csv')
piu_df = pd.read_csv('piu_geocoded.csv')

# extract all CUG numbers from officers cug column
def extract_numbers(cug_str):
    return re.findall(r'8\d{9}|[6-9]\d{9}', str(cug_str))

officers_df['numbers'] = officers_df['cug'].apply(extract_numbers)

# explode so each number gets its own row
officers_exploded = officers_df.explode('numbers')
officers_exploded = officers_exploded.dropna(subset=['numbers'])

# clean piu_contact
piu_df['piu_contact_clean'] = piu_df['piu_contact'].astype(str).str.extract(r'(\d{10})')[0]

# join
merged = piu_df.merge(
    officers_exploded[['name', 'designation', 'email', 'numbers']],
    left_on='piu_contact_clean',
    right_on='numbers',
    how='left'
)

print(f"matched: {merged['email'].notna().sum()}/{len(merged)}")
print(merged[merged['email'].notna()][['piu_city', 'piu_contact', 'name', 'email']].head(10).to_string())

merged.to_csv('piu_enriched.csv', index=False)

matched: 189/199
        piu_city   piu_contact                       name                                            email
0      Kamareddy  8.130006e+09       Sh. P. Nageswara Rao           pnrao@nhai.org , piukamareddy@nhai.org
1     Mancherial  8.130006e+09   Sh. K.N. Ajay Mani Kumar     knajaymani@nhai.org , piumancherial@nhai.org
2      Ahmedabad  8.130006e+09             Sh. C.M. Sinha                                     ahd@nhai.org
3    Ahilyanagar  8.130006e+09  Sh. Milindkumar S. Wabale       Ahmednagar@nhai.org , milindkumar@nhai.org
4          Ajmer  8.130006e+09       Sh. Hari Singh Geela              ajmer@nhai.org , harisingh@nhai.org
5         Jaipur  8.130006e+09        Sh. Ajay Kumar Arya                 ajayarya@nhai.org , jai@nhai.org
6         Kanpur  8.130006e+09           Sh. Pankaj Yadav              knp@nhai.org , pankajyadav@nhai.org
7         Jhansi  8.130006e+09            Sh. Manoj Kumar              manoj.kumar@nhai.org , jha@nhai.org
8  Amravati (MH)  8.

In [9]:
# check what designations the RO officers have
ro_officers = officers_df[officers_df['designation'].str.contains('Regional Officer|Chief General Manager', na=False)]
print(ro_officers[['name', 'designation', 'email']].to_string())

                                 name                             designation                                          email
36            Sh. Shrawan Kumar Singh  Chief General Manager/Regional Officer                              robhopal@nhai.org
82              Sh. Pradeep Kumar Lal      General Manager & Regional Officer        roodisha@nhai.org , pradeeplal@nhai.org
100                  Sh. Rakesh Kumar  Chief General Manager/Regional Officer                          rochandigarh@nhai.org
127              Sh. Virender Sambyal      General Manager & Regional Officer  virendersambyal@nhai.org , rochennai@nhai.org
151       Sh. Dinesh Kumar Chaturvedi      General Manager & Regional Officer      routtarakhand@nhai.org , dineshc@nhai.org
166                 Sh. Mohammad Safi  Chief General Manager/Regional Officer                          mohammadsafi@nhai.org
197                  Sh. Pardeep Atri      General Manager & Regional Officer      pardeepatri@nhai.org , rogujarat@nhai.org


In [10]:
import pandas as pd
import json
import re

officers_df = pd.read_csv('nhai_officers.csv')
piu_df = pd.read_csv('piu_enriched.csv')

# build RO email lookup from ro_name → email
# ro_name like 'RO-Hyderabad' → 'rohyderabad@nhai.org'
ro_officers = officers_df[officers_df['designation'].str.contains(
    'Regional Officer|Chief General Manager', na=False
)]

# extract city-based RO emails (ro* pattern)
def get_ro_email(email_str):
    if pd.isna(email_str):
        return ''
    emails = [e.strip() for e in str(email_str).split(',')]
    for e in emails:
        if e.startswith('ro'):
            return e
    return emails[0]

ro_officers = ro_officers.copy()
ro_officers['ro_email'] = ro_officers['email'].apply(get_ro_email)

# build lookup: normalize city name → email
# 'rohyderabad@nhai.org' → key 'hyderabad'
ro_lookup = {}
for _, row in ro_officers.iterrows():
    email = row['ro_email']
    if email.startswith('ro'):
        city = email.replace('ro', '').split('@')[0]
        ro_lookup[city] = email

print("RO lookup:", ro_lookup)

# match ro_name to ro_lookup
# 'RO-Hyderabad' → 'hyderabad' → 'rohyderabad@nhai.org'
def get_ro_email_for_piu(ro_name):
    if pd.isna(ro_name):
        return ''
    city = ro_name.replace('RO-', '').lower().strip()
    # handle special cases
    city = city.replace(' ', '').replace('(hr)', '').replace('(pb)', '').replace('(up)', '')
    return ro_lookup.get(city, '')

piu_df['ro_email'] = piu_df['ro_name'].apply(get_ro_email_for_piu)

# final email with fallback chain: PIU email → RO email → blank
def get_best_email(row):
    if pd.notna(row.get('email')) and str(row.get('email', '')).strip():
        emails = [e.strip() for e in str(row['email']).split(',')]
        for e in emails:
            parts = e.split('@')[0] if '@' in e else ''
            if len(parts) <= 10:
                return e
        return emails[0]
    # fallback to RO
    if row.get('ro_email'):
        return row['ro_email']
    return ''

piu_df['best_email'] = piu_df.apply(get_best_email, axis=1)

print(f"PIU email: {piu_df['best_email'].str.contains('@').sum()}")
print(f"no email: {(piu_df['best_email'] == '').sum()}")

# save
piu_list = piu_df[['piu_city', 'piu_contact', 'ro_name', 'ro_contact',
                    'lat', 'lng', 'name', 'designation', 'best_email']].to_dict('records')

with open('frontend/public/piu_locations.json', 'w') as f:
    json.dump(piu_list, f)

print(f"saved {len(piu_list)} PIUs")

RO lookup: {'bhopal': 'robhopal@nhai.org', 'odisha': 'roodisha@nhai.org', 'chandigarh': 'rochandigarh@nhai.org', 'chennai': 'rochennai@nhai.org', 'uttarakhand': 'routtarakhand@nhai.org', 'gujarat': 'rogujarat@nhai.org', 'ghy': 'roghy@nhai.org', 'hyderabad': 'rohyderabad@nhai.org', 'jaipur': 'rojaipur@nhai.org', 'jammu': 'rojammu@nhai.org', 'kolkatta': 'rokolkatta@nhai.org', 'madurai': 'romadurai@nhai.org', 'mumbai': 'romumbai@nhai.org', 'nagpur': 'ronagpur@nhai.org', 'patna': 'ropatna@nhai.org', 'raipur': 'roraipur@nhai.org', 'ranchi': 'roranchi@nhai.org', 'varansi': 'rovaransi@nhai.org', 'westup': 'rowestup@nhai.org', 'vijayawada': 'rovijayawada@nhai.org'}
PIU email: 196
no email: 3
saved 199 PIUs
